<a href="https://colab.research.google.com/github/LuciaKajanova/dspracticum25_flowers_team/blob/simpleTokenizationZakony/SimpleTokenizationMergedZakony.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Transformer

You may want to refer directly to [the git repo](https://github.com/karpathy/ng-video-lecture) instead though.

In [1]:





import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

# for saving the model
import os, json
SAVE_DIR = "./final_model"
MODEL_PATH = os.path.join(SAVE_DIR, "model_final.pt")
META_PATH = os.path.join(SAVE_DIR, "meta.json")
os.makedirs(SAVE_DIR, exist_ok=True)

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
# with open('/content/merged_zakony.md', 'r', encoding='utf-8') as f:
#   text = f.read()
                    ################ TAHÁNÍ PŘÍMO Z GITHUBU ################
!wget -O merged_zakony.md https://raw.githubusercontent.com/LuciaKajanova/dspracticum25_flowers_team/main/merged_zakony.md
with open('merged_zakony.md', 'r', encoding='utf-8') as f:
    text = f.read()
                    ################ TAHÁNÍ PŘÍMO Z GITHUBU ################


################ Přidání tokenizace ################
import tiktoken

encoding = tiktoken.get_encoding("o200k_base")

encode = lambda s: encoding.encode(s)
decode = lambda l: encoding.decode(l)

data = torch.tensor(encode(text), dtype=torch.long)
vocab_size = encoding.n_vocab
################ Přidání tokenizace ################

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

################ upravení generovani textu pro novou tokenizaci ################
# generate from the model
seed_text = "\n"  # nebo jakýkoliv začátek textu, např. "§ 140"
start = torch.tensor([encode(seed_text)], dtype=torch.long, device=device)

# generování textu
generated = m.generate(start, max_new_tokens=500)[0].tolist()
print(decode(generated))



--2025-10-26 17:50:37--  https://raw.githubusercontent.com/LuciaKajanova/dspracticum25_flowers_team/main/merged_zakony.md
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2591753 (2.5M) [text/plain]
Saving to: ‘merged_zakony.md’

merged_zakony.md    100%[===================>]   2.47M  --.-KB/s    in 0.04s   

2025-10-26 17:50:38 (56.2 MB/s) - ‘merged_zakony.md’ saved [2591753/2591753]

26.003795 M parameters
step 0: train loss 12.3623, val loss 12.3641
step 100: train loss 6.6861, val loss 6.5245
step 200: train loss 6.2583, val loss 6.1004
step 300: train loss 5.8247, val loss 5.7592
step 400: train loss 5.3683, val loss 5.3983
step 500: train loss 5.0004, val loss 5.1344
step 600: train loss 4.6861, val loss 4.8782
step 700: train loss 4.4758, val loss 4.69

In [2]:
import json

def save_meta():
    meta = {
        "block_size": block_size,
        "n_embd": n_embd,
        "n_head": n_head,
        "n_layer": n_layer,
        "dropout": dropout,
        "vocab_size": encoding.n_vocab,  # počet tokenů z tiktoken
    }
    with open(META_PATH, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

save_meta()
torch.save(model.state_dict(), MODEL_PATH)
print(f"Saved final model to {MODEL_PATH} and meta to {META_PATH}")


Saved final model to ./final_model/model_final.pt and meta to ./final_model/meta.json


## Using the saved model

The following code recreates the model and reads saved weights.

In [5]:
# - Assumes you trained & saved with:
#   - MODEL:  ./final_model/model_final.pt
#   - META:   ./final_model/meta.json

import torch
import torch.nn as nn
from torch.nn import functional as F
import os, json

SAVE_DIR = "./final_model"
MODEL_PATH = os.path.join(SAVE_DIR, "model_final.pt")
META_PATH = os.path.join(SAVE_DIR, "meta.json")

device = 'cuda' if torch.cuda.is_available() else 'cpu'

if not (os.path.exists(MODEL_PATH) and os.path.exists(META_PATH)):
    raise FileNotFoundError(f"Missing model or meta file in {SAVE_DIR}")

with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

block_size = meta["block_size"]
n_embd     = meta["n_embd"]
n_head     = meta["n_head"]
n_layer    = meta["n_layer"]
dropout    = meta["dropout"]

import tiktoken

# z předchozího tréninku
# encoding = tiktoken.get_encoding("o200k_base")  # nebo jiný encoding, který jsi použil
vocab_size = encoding.n_vocab  # počet tokenů
encode = encoding.encode
decode = encoding.decode


# ---------- model definition ----------
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    def __init__(self, n_embd_):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd_, 4 * n_embd_),
            nn.ReLU(),
            nn.Linear(4 * n_embd_, n_embd_),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd_, n_head_):
        super().__init__()
        head_size = n_embd_ // n_head_
        self.sa = MultiHeadAttention(n_head_, head_size)
        self.ffwd = FeedFoward(n_embd_)
        self.ln1 = nn.LayerNorm(n_embd_)
        self.ln2 = nn.LayerNorm(n_embd_)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ---------- load weights ----------
model = BigramLanguageModel().to(device)
state = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state)
model.eval()



################ upraveí generate pro generovaní pri nove tokenizaci ################
# ---------- generate ----------
seed_text = "Zákon č. 246/1992 Sb. § 5"  # you can add a prompt here if you want
###if else neni potreba pro novou tokenizaci s
start = torch.tensor([encode(seed_text)], dtype=torch.long, device=device)

generated = model.generate(start, max_new_tokens=500)[0].tolist()
print(decode(generated))


Zákon č. 246/1992 Sb. § 5 obdobných hospoda pro rozhodnutí záloření prezidentPro
nveřejníka, má vlastnost mohl na náklady zakanoupení, že nehřebuje. Je-li nada mu zapotřebí lhůtováni
povinnosti pod něj důvodů statutárního důbě tohoto státu Evropského parlamentů je k jeho států manželů a některýjmy
za důvod, neu existenci vytknístění mimo listinu vznikládatném jejího cenného papírem, je po určena nejbližiku peněji, ve věc za to, že
dvrhovuje-li plnění i společením pro příkaz právo na postavu, neujednanou.
§ 1607
Vde-li emitent sloudit jmenovací zneužite­ |
| více právní povin společnosti věku 12 a jiného zprostí strany odstraní kraje a  |
| vynaloženém místnosti jako hradíám |
| c)   cenné papírem může podepsabyvat se předešný si základně pečnéhoností na návrvynaložení nezbytného.
§ 655
(1)   Má-li-li ke snášťmenací, že soud na právo rodšetiní uloží k tomu
zástavce prostředky volby
o této, že tato další vlastník navrhnout, nejméně za jeden sporce omylučitele neposta platí, do jeho statu

In [4]:
import os
MODEL_PATH = "./final_model/model_final.pt"
print(os.path.getsize(MODEL_PATH) / 1024**2, "MB")



99.29887676239014 MB
